# Heterogeneity Review

This notebook presents the repeated-split heterogeneity results: the BLP test, sorted GATEs, CLAN summary, and permutation-adjusted importance.

## Outputs Produced By This Notebook

This notebook runs the repeated-split heterogeneity script and writes the final summary outputs to `outputs/heterogeneity_results/`.

Key saved files are:
- `repeated_split_blp_summary.csv`
- `repeated_split_sorted_gate_summary.csv`
- `repeated_split_sorted_gate_plot.png`
- `repeated_split_clan_summary.csv`
- `repeated_split_clan_table.csv`
- `repeated_split_settings.csv`
- `permutation_importance_summary.csv`
- `permutation_importance_plot.png`

In [ ]:
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

import matplotlib as mpl
mpl.rcParams['figure.dpi'] = 180
mpl.rcParams['savefig.dpi'] = 300

from pathlib import Path
import importlib.util

_cwd = Path.cwd().resolve()
_cands = [_cwd, _cwd.parent]
PROJECT_ROOT = next((p for p in _cands if (p / "MODELS" / "03_heterogeneity_analysis.py").exists()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate project root")

MODELS_DIR = PROJECT_ROOT / "MODELS"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
HETERO_DIR = OUTPUT_DIR / "heterogeneity_results"

def load_module(module_filename: str, module_name: str):
    module_path = MODELS_DIR / module_filename
    spec = importlib.util.spec_from_file_location(module_name, module_path)
    if spec is None or spec.loader is None:
        raise ImportError(f"Could not load {module_path}")
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

heterogeneity_module = load_module("03_heterogeneity_analysis.py", "heterogeneity_analysis")
PROJECT_ROOT.relative_to(PROJECT_ROOT.parent)


## Run The Heterogeneity Script

The script below generates the repeated-split heterogeneity results used in this notebook.

In [ ]:
import os

os.environ.setdefault("HET_N_SPLITS", "100")
os.environ.setdefault("HET_CF_TREES", "400")
os.environ.setdefault("HET_N_PERM", "50")
os.environ.setdefault("HET_PERM_CF_TREES", "200")
heterogeneity_module.main()


## 1. Repeated-Split Formal Inference

These outputs summarize the formal repeated-split heterogeneity results.


### BLP Summary

This table reports the repeated-split BLP-style omnibus heterogeneity test. The key quantity is whether the heterogeneity term remains clearly above zero across splits.


In [ ]:
import pandas as pd
pd.read_csv(HETERO_DIR / "repeated_split_blp_summary.csv")


### Repeated-Split Sorted GATE

This table summarizes how the estimated treatment effect changes across effect-sorted groups under repeated sample splitting. The figure underneath gives the same result in a more visual form.


In [ ]:
pd.read_csv(HETERO_DIR / "repeated_split_sorted_gate_summary.csv")


In [ ]:
from IPython.display import Image, display
display(Image(filename=str(HETERO_DIR / "repeated_split_sorted_gate_plot.png")))


### Repeated-Split CLAN

This is the main group-characterization table. It compares the most affected group (Q1) with the least affected group (Q4) using the repeated-split summary, so it is the version that should be cited in the report.

In [ ]:
pd.read_csv(HETERO_DIR / "repeated_split_clan_table.csv")

### Permutation-Adjusted Importance

This table reports the permutation-based importance exercise, which is more conservative than raw forest importance. The plot underneath makes the relative magnitudes easier to compare.


In [ ]:
pd.read_csv(HETERO_DIR / "permutation_importance_summary.csv")


In [ ]:
display(Image(filename=str(HETERO_DIR / "permutation_importance_plot.png")))


## 2. Interpreting The Repeated-Split Results

The BLP results test whether heterogeneity is present, the sorted GATE results summarize the ranked effect gap, and the CLAN table describes the most- and least-affected groups. The permutation-adjusted importance exercise is a stricter robustness check.